In [0]:
# Importaciones
import pandas as pd
from pyspark.sql.functions import lit

In [0]:
# Descargar y combinar los dos archivos CSV
URL_RED   = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
URL_WHITE = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"

df_red_pd   = pd.read_csv(URL_RED,   sep=";")
df_white_pd = pd.read_csv(URL_WHITE, sep=";")

# Agregar columna de tipo de vino
df_red_pd["wine_type"]   = "red"
df_white_pd["wine_type"] = "white"

# Combinar en un solo DataFrame de pandas
df_pd = pd.concat([df_red_pd, df_white_pd], ignore_index=True)

print(f"Filas totales: {len(df_pd)}")
print(f"Columnas: {list(df_pd.columns)}")
print(df_pd.head())

Filas totales: 6497
Columnas: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality', 'wine_type']
   fixed acidity  volatile acidity  citric acid  ...  alcohol  quality  wine_type
0            7.4              0.70         0.00  ...      9.4        5        red
1            7.8              0.88         0.00  ...      9.8        5        red
2            7.8              0.76         0.04  ...      9.8        5        red
3           11.2              0.28         0.56  ...      9.8        6        red
4            7.4              0.70         0.00  ...      9.4        5        red

[5 rows x 13 columns]


In [0]:
# Convertir a Spark DataFrame y guardar como Delta
df_spark = spark.createDataFrame(df_pd)

# Renombrar columnas: los espacios causan problemas en SQL
new_cols = [c.replace(" ", "_") for c in df_spark.columns]
df_spark = df_spark.toDF(*new_cols)

df_spark.printSchema()

# Guardar como tabla Delta permanente (disponible para todos los notebooks)
df_spark.write.format("delta").mode("overwrite").saveAsTable("wine_quality_raw")

print("✅ Tabla 'wine_quality_raw' creada exitosamente.")

root
 |-- fixed_acidity: double (nullable = true)
 |-- volatile_acidity: double (nullable = true)
 |-- citric_acid: double (nullable = true)
 |-- residual_sugar: double (nullable = true)
 |-- chlorides: double (nullable = true)
 |-- free_sulfur_dioxide: double (nullable = true)
 |-- total_sulfur_dioxide: double (nullable = true)
 |-- density: double (nullable = true)
 |-- pH: double (nullable = true)
 |-- sulphates: double (nullable = true)
 |-- alcohol: double (nullable = true)
 |-- quality: long (nullable = true)
 |-- wine_type: string (nullable = true)

✅ Tabla 'wine_quality_raw' creada exitosamente.


In [0]:
# Verificar que la tabla quedó bien
display(spark.sql("SELECT * FROM wine_quality_raw LIMIT 10"))

fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality,wine_type
6.3,0.35,0.3,5.7,0.035,8.0,97.0,0.9927,3.27,0.41,11.0,7,white
6.4,0.23,0.39,1.8,0.032,23.0,118.0,0.9912,3.32,0.5,11.8,6,white
5.8,0.36,0.38,0.9,0.037,3.0,75.0,0.9904,3.28,0.34,11.4,4,white
6.9,0.115,0.35,5.4,0.048,36.0,108.0,0.9939,3.32,0.42,10.2,6,white
6.9,0.29,0.4,19.45,0.043,36.0,156.0,0.9996,2.93,0.47,8.9,5,white
6.9,0.28,0.4,8.2,0.036,15.0,95.0,0.9944,3.17,0.33,10.2,5,white
7.2,0.29,0.4,13.6,0.045,66.0,231.0,0.9977,3.08,0.59,9.6,6,white
6.2,0.24,0.35,1.2,0.038,22.0,167.0,0.9912,3.1,0.48,10.6,6,white
6.9,0.29,0.4,19.45,0.043,36.0,156.0,0.9996,2.93,0.47,8.9,5,white
6.9,0.32,0.26,8.3,0.053,32.0,180.0,0.9965,3.25,0.51,9.2,6,white
